In [ ]:
import pyam
import ixmp4

In [ ]:
platform = ixmp4.Platform("scenariocompass-dev")

# Remove 2110 data

In [ ]:
data_2110 = platform.iamc.tabulate(year=2110)

In [ ]:
for _data in data_2110.groupby(
    ["model", "scenario", "version"]
):
    platform.runs.get(*_data[0]).iamc.remove(_data[1])

# Zero-capacity assignment for historical periods

In [ ]:
#"Capacity|Electricity|*"
variables = "Primary Energy|Biomass"

In [ ]:
df = pyam.read_ixmp4(platform, variable=variables)

In [ ]:
df.filter(year=range(1990, 2021), region="World").plot()

In [ ]:
data = df.filter(year=range(1990, 2020), region="World").plot()

In [ ]:
data = data[data.value.le(0.001)]
data.year.unique()

In [ ]:
for _data in data.groupby(
    ["model", "scenario"]
):
    platform.runs.get(*_data[0]).iamc.remove(_data[1])

In [ ]:
list()

In [ ]:
platform.iamc.variables.tabulate()

# Reset "n/a" meta indicators to None

In [ ]:
meta = platform.meta.tabulate()

In [ ]:
meta = meta[meta.value == "n/a"]

In [ ]:
for _meta in meta.groupby(
    ["model", "scenario", "version"]
):
    run = platform.runs.get(*_meta[0])
    for key in _meta[1].key:
        run.meta[key] = None

# Fix meta indicator names

In [ ]:
meta = platform.meta.tabulate()

In [ ]:
[x for x in meta.value.unique() if isinstance(x, str)]

In [ ]:
meta_key_fix = {
    "Reason of Concern|Sustainable Bioenergy Use|World": "Reason For Concern|Sustainable Bioenergy Use|World",
    "Plausibility Vetting|Hydropower Capacity|2030": "Plausibility Vetting|Hydropower Capacity|World|2030",
    "Plausibility Vetting|Nuclear Capacity|2030": "Plausibility Vetting|Nuclear Capacity|World|2030",
    "Plausibility Vetting|Onshore Wind Capacity|2025": "Plausibility Vetting|Onshore Wind Capacity|World|2025",
    "Plausibility Vetting|Onshore Wind Capacity|2030": "Plausibility Vetting|Onshore Wind Capacity|World|2030",
}

In [ ]:
for _meta in meta.groupby(
    ["model", "scenario", "version"]
):
    run = platform.runs.get(*_meta[0])
    for old, new in meta_key_fix.items():
        if old in _meta[1].key.values:
            run.meta[new] = _meta[1][_meta[1].key == old].value.iloc[0]
            run.meta[old] = None

In [ ]:
meta.key.unique()